# MLflow RAG Generation Evaluation on Amazon Bedrock

This notebook demonstrates how to evaluate the **generation step of a RAG pipeline** using **MLflow's GenAI evaluation framework**. Retrieved context is supplied by the dataset (no retriever is run), so the metrics measure how faithfully a LLM grounds its answer in that context — not how well a retriever finds it. All metrics, parameters, and artifacts are tracked in MLflow so you can compare models side-by-side.

## Scope

- **In scope:** generator quality (faithfulness, groundedness, hallucination, relevance, fluency, correctness, safety) holding retrieval constant.
- **Out of scope:** retrieval quality (recall@k, precision@k, MRR, context precision/recall). There is no retriever in this pipeline — context comes from the dataset fixture.

## What You'll Learn

- How to shape a HuggingFace RAG dataset for `mlflow.genai.evaluate()` (`{inputs: {question, context}, expectations: {...}}`)
- How to register **18 scorers** across five categories in a single run
- How to call Bedrock directly via the Converse API as the generator model (with context injected into the prompt)
- How to use **Bedrock Claude as the judge**
- How to compare multiple model runs via `mlflow.search_runs`

## Scorer Summary

| Category | Scorers |
|----------|---------|
| MLflow built-in LLM-as-Judge | `RelevanceToQuery`, `Equivalence`, `Fluency` |
| MLflow Guidelines (LLM-as-Judge) | `answer_groundedness`, `answer_completeness` |
| Custom `make_judge` | `factual_consistency`, `professionalism`, `correctness` |
| DeepEval via Bedrock | `Faithfulness`, `AnswerRelevancy`, `ContextualRelevancy`, `Hallucination`, `Toxicity`, `Bias` |
| Code-based | `exact_match`, `is_concise`, `word_overlap`, `response_length` |


## Prerequisites

Set up a Python environment and install the requirements in `requirements.txt`. For example, if you are using `uv`, run:

```
uv venv .venv
source .venv/bin/activate
uv pip install -r requirements.txt
```

The requirements.txt contains the following dependencies:
```
mlflow>=3.8.0
sagemaker-mlflow>=0.3.0
deepeval
datasets
boto3>=1.42.4
botocore[crt]
pandas
nest_asyncio
litellm
python-dotenv
```

## 1. Install Dependencies

In [1]:
!pip install -r requirements.txt --quiet

## 2. Configure AWS, Bedrock, and MLflow

We initialize the boto3 session and Bedrock runtime client, plus a `Config` with retry/backoff to survive Bedrock throttling.

The LLM-as-Judge scorers (MLflow built-in, `make_judge`, DeepEval) call Bedrock through LiteLLM. LiteLLM does **not** use the boto3 default credential chain — it reads AWS credentials from environment variables — so we also export the session's credentials below.

By default, MLflow writes runs to a **local store** (`./mlruns`). To track runs in a serverless SageMaker MLflow App instead, set `MLFLOW_TRACKING_URI` to the App ARN before running this cell.

In [2]:
import os
import logging
import boto3
from botocore.config import Config
import nest_asyncio

nest_asyncio.apply()

# -----------------------------------------------------------------------------
# Load environment variables from a local .env file (if present).
# Copy .env.example to .env and fill in AWS_REGION + AWS_PROFILE
# -----------------------------------------------------------------------------
from dotenv import load_dotenv, dotenv_values

_dotenv_values = dotenv_values(".env")
load_dotenv(dotenv_path=".env", override=True)

# Suppress noisy LiteLLM logs used by the judge / DeepEval scorers
logging.getLogger("LiteLLM").setLevel(logging.WARNING)

# -----------------------------------------------------------------------------
# Bedrock client
# -----------------------------------------------------------------------------
config = Config(
    retries={
        "max_attempts": 10,
        "mode": "standard",
    }
)

# AWS Configuration
AWS_REGION = os.getenv("AWS_REGION", os.getenv("AWS_DEFAULT_REGION", "us-east-1"))
AWS_PROFILE = os.getenv("AWS_PROFILE")

session_kwargs = {"region_name": AWS_REGION}
if AWS_PROFILE:
    session_kwargs["profile_name"] = AWS_PROFILE
    cred_source = f"profile={AWS_PROFILE}"
else:
    cred_source = "default chain"

try:
    # One session reused by Bedrock, STS, S3, IAM, and SageMaker
    session = boto3.Session(**session_kwargs)
    region = session.region_name or "us-east-1"

    bedrock = session.client(
        service_name="bedrock-runtime",
        region_name=region,
        config=config,
    )

    # Test if credentials are valid. Print only the last 4 digits of the
    # account id — enough to spot misrouted creds without leaking the full
    # account to notebook output shared in screenshots.
    identity = session.client("sts").get_caller_identity()
    account_suffix = identity["Account"][-4:]
    print(
        f"✅ Initialized AWS clients for account ****{account_suffix} "
        f"(source={cred_source}, region={region})"
    )

except Exception as e:
    print(f"\n❌ Failed to initialize AWS clients: {e}")
    print(f"   Please check your AWS credentials and configuration")
    raise

# LiteLLM (used by MLflow LLM-as-Judge scorers + DeepEval) reads AWS credentials
# from environment variables rather than the boto3 default chain, so we export
# the session's frozen credentials explicitly.
_creds = session.get_credentials().get_frozen_credentials()
os.environ["AWS_ACCESS_KEY_ID"] = _creds.access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = _creds.secret_key
os.environ["AWS_REGION"] = region
os.environ["AWS_DEFAULT_REGION"] = region
if _creds.token:
    os.environ["AWS_SESSION_TOKEN"] = _creds.token


# -----------------------------------------------------------------------------
# Bedrock models under evaluation
# -----------------------------------------------------------------------------
BEDROCK_MODELS = {
    "claude-sonnet-4-5": "us.anthropic.claude-sonnet-4-5-20250929-v1:0",
    "claude-haiku-4-5":  "us.anthropic.claude-haiku-4-5-20251001-v1:0",
}

# Judge model used by all LLM-as-Judge scorers. The `bedrock:/` prefix tells
# MLflow/LiteLLM to route calls through the Bedrock adapter.
JUDGE_MODEL = "bedrock:/us.anthropic.claude-sonnet-4-5-20250929-v1:0"

# -----------------------------------------------------------------------------
# Dataset
# -----------------------------------------------------------------------------
DATASET_NAME = "explodinggradients/ragas-wikiqa"
DATASET_SPLIT = "train"
SAMPLE_SIZE = 5  # small default so the notebook runs quickly end-to-end

print(f"Bedrock client initialized for region: {region}")
print(f"Judge model:    {JUDGE_MODEL}")
print(f"Models to eval: {list(BEDROCK_MODELS.keys())}")

✅ Initialized AWS clients for account ****0000 (source=profile=<redacted>, region=us-east-1)
Bedrock client initialized for region: us-east-1
Judge model:    bedrock:/us.anthropic.claude-sonnet-4-5-20250929-v1:0
Models to eval: ['claude-sonnet-4-5', 'claude-haiku-4-5']


In [3]:
# -----------------------------------------------------------------------------
# Bootstrap AWS resources required by the SageMaker MLflow App:
#   1. S3 bucket that holds MLflow artifacts
#   2. IAM service role that the MLflow App assumes
#
# Both operations are idempotent — safe to re-run. If the bucket or role
# already exists they are reused as-is. Skip this cell entirely if you plan
# to use the local ./mlruns store.
# -----------------------------------------------------------------------------
import json as _json
from botocore.exceptions import ClientError


def _mask_account(text: str, account_id: str) -> str:
    """Return `text` with any occurrence of `account_id` replaced by `****<last4>`.

    ARNs and other strings printed from this cell embed the caller's account
    number; we mask it so notebook output is safe to share in screenshots.
    """
    return text.replace(account_id, f"****{account_id[-4:]}")


# Resolve the account once and keep only the last 4 digits in plaintext.
_account_id = session.client("sts").get_caller_identity()["Account"]
_account_suffix = _account_id[-4:]

# Edit these to match what you want the MLflow App to use.
# The bucket name must be globally unique; the role name is account-scoped.
# Including a short account suffix (last 4 digits) keeps the bucket name unique
# across accounts without leaking the full id.
MLFLOW_APP_NAME      = "bedrock-evals"
MLFLOW_S3_BUCKET     = f"mlflow-artifacts-{_account_suffix}-{region}"
MLFLOW_S3_PREFIX     = "mlflow-artifacts"
MLFLOW_APP_ROLE_NAME = "sagemaker-mlflow-app-servicerole"

s3 = session.client("s3", region_name=region)
iam = session.client("iam")


def ensure_s3_bucket(bucket: str) -> str:
    """Create an S3 bucket if it doesn't already exist. Returns the bucket name.

    us-east-1 requires omitting the LocationConstraint, every other region
    requires it — so we branch on `region`.
    """
    try:
        s3.head_bucket(Bucket=bucket)
        print(f"Reusing existing S3 bucket: s3://{bucket}")
        return bucket
    except ClientError as e:
        code = e.response["Error"]["Code"]
        if code not in ("404", "NoSuchBucket", "NotFound"):
            raise

    kwargs = {"Bucket": bucket}
    if region != "us-east-1":
        kwargs["CreateBucketConfiguration"] = {"LocationConstraint": region}
    s3.create_bucket(**kwargs)
    print(f"Created S3 bucket: s3://{bucket}")
    return bucket


def ensure_mlflow_app_role(role_name: str, bucket: str, prefix: str) -> str:
    """Create (or reuse) the IAM role that the MLflow App assumes. Returns the role ARN.

    Permissions mirror the reference CDK stack: S3 access scoped to the
    artifact bucket, plus SageMaker model-registry + sagemaker-mlflow actions.
    """
    trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "sagemaker.amazonaws.com"},
                "Action": "sts:AssumeRole",
            }
        ],
    }

    inline_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Sid": "MlflowArtifactAccess",
                "Effect": "Allow",
                "Action": [
                    "s3:GetObject",
                    "s3:PutObject",
                    "s3:DeleteObject",
                    "s3:ListBucket",
                    "s3:GetBucketLocation",
                ],
                "Resource": [
                    f"arn:aws:s3:::{bucket}",
                    f"arn:aws:s3:::{bucket}/{prefix}/*",
                ],
            },
            {
                "Sid": "MlflowAppAccess",
                "Effect": "Allow",
                "Action": ["sagemaker-mlflow:*"],
                "Resource": "*",
            },
            {
                "Sid": "ModelRegistryAccess",
                "Effect": "Allow",
                "Action": [
                    "sagemaker:AddTags",
                    "sagemaker:CreateModelPackageGroup",
                    "sagemaker:CreateModelPackage",
                    "sagemaker:DescribeModelPackageGroup",
                    "sagemaker:UpdateModelPackage",
                ],
                "Resource": "*",
            },
        ],
    }

    try:
        role = iam.get_role(RoleName=role_name)["Role"]
        print(f"Reusing existing IAM role: {_mask_account(role['Arn'], _account_id)}")
    except ClientError as e:
        if e.response["Error"]["Code"] != "NoSuchEntity":
            raise
        role = iam.create_role(
            RoleName=role_name,
            AssumeRolePolicyDocument=_json.dumps(trust_policy),
            Description="Service role assumed by the SageMaker MLflow App",
        )["Role"]
        print(f"Created IAM role: {_mask_account(role['Arn'], _account_id)}")

    # put_role_policy is idempotent — refreshes permissions on re-run
    iam.put_role_policy(
        RoleName=role_name,
        PolicyName="mlflow-app-inline",
        PolicyDocument=_json.dumps(inline_policy),
    )
    return role["Arn"]


MLFLOW_S3_BUCKET    = ensure_s3_bucket(MLFLOW_S3_BUCKET)
MLFLOW_ARTIFACT_S3  = f"s3://{MLFLOW_S3_BUCKET}/{MLFLOW_S3_PREFIX}"
MLFLOW_APP_ROLE_ARN = ensure_mlflow_app_role(MLFLOW_APP_ROLE_NAME, MLFLOW_S3_BUCKET, MLFLOW_S3_PREFIX)

# Print the masked ARN for the log — the unmasked value stays in
# MLFLOW_APP_ROLE_ARN for downstream cells that need to pass it to SageMaker.
print(f"Artifact URI:  {MLFLOW_ARTIFACT_S3}")
print(f"App role ARN:  {_mask_account(MLFLOW_APP_ROLE_ARN, _account_id)}")

Reusing existing S3 bucket: s3://mlflow-artifacts-8683-us-east-1
Reusing existing IAM role: arn:aws:iam::****8683:role/sagemaker-mlflow-app-servicerole
Artifact URI:  s3://mlflow-artifacts-8683-us-east-1/mlflow-artifacts
App role ARN:  arn:aws:iam::****8683:role/sagemaker-mlflow-app-servicerole


In [4]:
# -----------------------------------------------------------------------------
# MLflow tracking backend
# -----------------------------------------------------------------------------
# Default: local MLflow store at ./mlruns — runs end-to-end with no cloud setup.
#
# To use a serverless SageMaker-managed MLflow App, the previous cell already
# bootstrapped the S3 bucket and IAM role. On first run this cell creates the
# MLflow App; on subsequent runs it describes-and-reuses the existing one.
#
# Requires boto3 >= 1.42.4 (ships the `create_mlflow_app` API, GA Dec 2025).
# Docs: https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html
# -----------------------------------------------------------------------------
import time
from botocore.exceptions import ClientError

EXPERIMENT_NAME = "bedrock-llm-evaluation"

# Reuse the boto3 session created in the AWS-config cell
sagemaker_client = session.client("sagemaker", region_name=region)
sts_client = session.client("sts", region_name=region)

# If `_mask_account` / `_account_id` weren't defined by the bootstrap cell
# (e.g. this cell is run standalone), resolve them here so ARN prints can be
# redacted without exposing the full account id in notebook output.
try:
    _account_id  # noqa: F821
    _mask_account  # noqa: F821
except NameError:
    _account_id = sts_client.get_caller_identity()["Account"]

    def _mask_account(text: str, account_id: str) -> str:
        """Replace any occurrence of `account_id` in `text` with `****<last4>`."""
        return text.replace(account_id, f"****{account_id[-4:]}")


def find_existing_mlflow_app(app_name: str) -> str | None:
    """Return the ARN of an MLflow App named `app_name` in this account, or None.

    `describe_mlflow_app` uses a deterministic ARN, but SageMaker assigns App
    ARNs with the app name in them — so we list + match. This is also the
    single source of truth when the account is at the 3-app quota and we
    must decide whether to reuse instead of create.
    """
    paginator = sagemaker_client.get_paginator("list_mlflow_apps")
    for page in paginator.paginate():
        for app in page.get("MlflowApps", []):
            if app.get("Name") == app_name:
                return app.get("Arn") or app.get("MlflowAppArn")
    return None


def list_mlflow_apps() -> list[dict]:
    """Return all MLflow Apps in this account (for quota diagnostics)."""
    out = []
    paginator = sagemaker_client.get_paginator("list_mlflow_apps")
    for page in paginator.paginate():
        out.extend(page.get("MlflowApps", []))
    return out


def _create_mlflow_app_with_role_propagation_retry(
    app_name: str, artifact_uri: str, role_arn: str
) -> dict:
    """Call create_mlflow_app, retrying while IAM propagates the role's trust policy.

    A freshly-created IAM role is not immediately assumable by SageMaker — IAM
    needs ~10-60s to propagate the trust relationship globally. During that
    window `create_mlflow_app` returns a ValidationException with the message
    "SageMaker is unable to perform: sts:AssumeRole on the role". We retry
    with exponential backoff on that specific error only.
    """
    max_attempts = 10
    delay = 10
    for attempt in range(1, max_attempts + 1):
        try:
            return sagemaker_client.create_mlflow_app(
                Name=app_name,
                ArtifactStoreUri=artifact_uri,
                RoleArn=role_arn,
                ModelRegistrationMode="AutoModelRegistrationEnabled",
            )
        except ClientError as e:
            code = e.response["Error"]["Code"]
            msg = e.response["Error"].get("Message", "")
            is_trust_propagation = (
                code == "ValidationException"
                and "sts:AssumeRole" in msg
            )
            if not is_trust_propagation or attempt == max_attempts:
                raise
            print(
                f"  IAM role trust not yet propagated (attempt {attempt}/{max_attempts}) — "
                f"waiting {delay}s..."
            )
            time.sleep(delay)
            delay = min(delay + 10, 30)
    raise RuntimeError("unreachable")


def deploy_or_describe_mlflow_app(app_name: str, artifact_uri: str, role_arn: str) -> str:
    """Create a SageMaker MLflow App (or describe an existing one) and return its ARN.

    Idempotent — safe to re-run. Polls until the App reaches the `Created` state.
    Reuses an existing App with the same name if one exists (even if its ARN
    format differs from the deterministic pattern).
    """
    existing_arn = find_existing_mlflow_app(app_name)
    if existing_arn:
        print(f"Reusing existing MLflow App: {_mask_account(existing_arn, _account_id)}")
        app_arn = existing_arn
    else:
        apps = list_mlflow_apps()
        if len(apps) >= 3:
            names = [a.get("Name") for a in apps]
            raise RuntimeError(
                f"Account already has {len(apps)} MLflow Apps (quota is 3): {names}. "
                f"Either delete one, rename MLFLOW_APP_NAME to match one of the existing apps, "
                f"or request a quota increase for 'Maximum number of MLflow apps allowed per account'."
            )
        print(f"Creating MLflow App '{app_name}' (artifacts: {artifact_uri})...")
        create_resp = _create_mlflow_app_with_role_propagation_retry(
            app_name=app_name,
            artifact_uri=artifact_uri,
            role_arn=role_arn,
        )
        # Prefer the ARN returned by create; fall back to the deterministic form
        app_arn = create_resp.get("Arn") or f"arn:aws:sagemaker:{region}:{_account_id}:mlflow-app/{app_name}"

    # Poll until the App is ready. Immediately after create_mlflow_app
    # SageMaker may briefly 404 on describe — tolerate a short window
    # of ResourceNotFound before giving up.
    terminal_ok = {"Created", "Updated"}
    terminal_fail = {"CreateFailed", "UpdateFailed", "DeleteFailed", "Deleted"}
    max_wait_seconds = 600   # total budget
    poll_every = 10
    not_found_grace = 60     # tolerate ResourceNotFound for this long post-create
    elapsed = 0
    while True:
        try:
            status = sagemaker_client.describe_mlflow_app(Arn=app_arn)["Status"]
        except ClientError as e:
            code = e.response["Error"]["Code"]
            if code in ("ResourceNotFound", "ValidationException") and elapsed < not_found_grace:
                print(f"  App not yet visible ({code}) — retrying...")
                time.sleep(poll_every)
                elapsed += poll_every
                continue
            raise
        if status in terminal_ok:
            break
        if status in terminal_fail:
            raise RuntimeError(f"MLflow App in terminal failure state: {status}")
        if elapsed >= max_wait_seconds:
            raise TimeoutError(f"MLflow App still {status} after {elapsed}s")
        print(f"  status={status} — waiting...")
        time.sleep(poll_every)
        elapsed += poll_every

    return app_arn


def get_mlflow_app_ui_url(app_arn: str, expires_in: int = 300) -> str:
    """Return a presigned URL for the MLflow App UI."""
    url = sagemaker_client.create_presigned_mlflow_app_url(
        Arn=app_arn,
        ExpiresInSeconds=expires_in,
        SessionExpirationDurationInSeconds=3600,
    )
    return url["AuthorizedUrl"]


# -----------------------------------------------------------------------------
# Resolve the tracking URI: if the previous cell populated MLFLOW_APP_NAME,
# MLFLOW_ARTIFACT_S3, and MLFLOW_APP_ROLE_ARN, deploy or reuse the App;
# otherwise, fall back to a local ./mlruns store.
# -----------------------------------------------------------------------------
if MLFLOW_APP_NAME and MLFLOW_ARTIFACT_S3 and MLFLOW_APP_ROLE_ARN:
    MLFLOW_TRACKING_URI = deploy_or_describe_mlflow_app(
        app_name=MLFLOW_APP_NAME,
        artifact_uri=MLFLOW_ARTIFACT_S3,
        role_arn=MLFLOW_APP_ROLE_ARN,
    )
    # MLFLOW_TRACKING_URI keeps the unmasked ARN for downstream API calls;
    # only the printed log line is masked.
    print(f"MLflow App ready: {_mask_account(MLFLOW_TRACKING_URI, _account_id)}")
    # The presigned URL itself is a short-lived secret and does not expose
    # the account id in plaintext (the ARN is base64-embedded inside the
    # JWT), so we render it as-is.
    print(f"UI URL (5-minute presigned): {get_mlflow_app_ui_url(MLFLOW_TRACKING_URI)}")
else:
    MLFLOW_TRACKING_URI = "./mlruns"
    print(f"Using local MLflow store: {MLFLOW_TRACKING_URI}")

os.environ["MLFLOW_TRACKING_URI"] = MLFLOW_TRACKING_URI

Creating MLflow App 'bedrock-evals' (artifacts: s3://mlflow-artifacts-8683-us-east-1/mlflow-artifacts)...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
  status=Creating — waiting...
MLflow App ready: arn:aws:

## 3. Load and Shape the Evaluation Dataset

We download the [explodinggradients/ragas-wikiqa](https://huggingface.co/datasets/explodinggradients/ragas-wikiqa) dataset and reshape it into the format that `mlflow.genai.evaluate()` expects:

```python
{
    "inputs":       { "question": ..., "context": ... },   # passed to predict_fn
    "expectations": { "expected_response": ..., "expected_facts": [...] },  # consumed by scorers
}
```

In [5]:
import json
import re
import pandas as pd
from datasets import load_dataset


def extract_facts(text: str) -> list[str]:
    """Split a ground-truth answer into up to 5 short factual claims.

    MLflow's `Correctness` scorer evaluates each fact individually, so we need
    short sentence-level claims rather than a single paragraph.
    """
    sentences = [s.strip() for s in re.split(r"[.!?]+", text) if len(s.strip()) > 10]
    return [s[:200] for s in sentences[:5]]


def normalize_context(value) -> str:
    """Coerce a dataset `context` field to a plain string.

    ragas-wikiqa stores context as a list-of-passages, and pandas surfaces it
    as a numpy ndarray — neither is JSON-serializable or usable directly as
    a Bedrock prompt, so we flatten to a single newline-joined string.
    """
    if isinstance(value, str):
        return value
    try:
        return "\n\n".join(str(v) for v in value)
    except TypeError:
        return str(value)


ds = load_dataset(DATASET_NAME, split=DATASET_SPLIT)
df = ds.to_pandas().head(SAMPLE_SIZE)

eval_data = [
    {
        "inputs": {
            "question": str(row["question"]),
            "context": normalize_context(row["context"]),
        },
        "expectations": {
            "expected_response": str(row["correct_answer"]),
            "expected_facts": extract_facts(str(row["correct_answer"])),
        },
    }
    for _, row in df.iterrows()
]

print(f"Loaded {len(eval_data)} samples from {DATASET_NAME}")
print("\nExample record:")
_preview = {
    **eval_data[0],
    "inputs": {
        **eval_data[0]["inputs"],
        "context": eval_data[0]["inputs"]["context"][:200] + "...",
    },
}
print(json.dumps(_preview, indent=2, default=str))

~/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 5 samples from explodinggradients/ragas-wikiqa

Example record:
{
  "inputs": {
    "question": "HOW AFRICAN AMERICANS WERE IMMIGRATED TO THE US",
    "context": "African immigration to the United States refers to immigrants to the United States who are or were nationals of modern African countries. The term African in the scope of this article refers to geogra..."
  },
  "expectations": {
    "expected_response": "As such, African immigrants are to be distinguished from African American people, the latter of whom are descendants of mostly West and Central Africans who were involuntarily brought to the United States by means of the historic Atlantic slave trade .",
    "expected_facts": [
      "As such, African immigrants are to be distinguished from African American people, the latter of whom are descendants of mostly West and Central Africans who were involuntarily brought to the United St"
    ]
  }
}


## 4. Define Code-Based Scorers

These are deterministic — no LLM calls. They run instantly and provide objective, reproducible metrics. MLflow tracks them alongside LLM-as-Judge metrics in the same run.

In [6]:
from mlflow.genai import scorer


@scorer
def exact_match(outputs: str, expectations: dict) -> bool:
    """Case-insensitive, whitespace-trimmed string equality."""
    return outputs.strip().lower() == expectations["expected_response"].strip().lower()


@scorer
def is_concise(outputs: str) -> bool:
    """True if the response is 100 words or fewer."""
    return len(outputs.split()) <= 100


@scorer
def word_overlap(outputs: str, expectations: dict) -> float:
    """Fraction of expected words that appear anywhere in the output."""
    output_words = set(outputs.lower().split())
    expected_words = set(expectations["expected_response"].lower().split())
    if not expected_words:
        return 0.0
    return len(output_words & expected_words) / len(expected_words)


@scorer
def response_length(outputs: str) -> int:
    """Word count of the model's response."""
    return len(outputs.split())

## 5. Define Custom LLM-as-Judge Scorers

`mlflow.genai.make_judge` lets us create custom evaluation criteria with Jinja2 prompt templates. The judge model (Bedrock Claude) scores each response against our rubric.

Template variables available:
- `{{ outputs }}` — the model's response
- `{{ expectations }}` — the full expectations dict

In [ ]:
from mlflow.genai import make_judge

# make_judge only allows top-level template vars (inputs, outputs, trace,
# expectations, conversation). Nested access like `{{ inputs.context }}` is
# rejected, so we pass the whole `inputs` dict — Claude can locate the
# `context` field inside it.
factual_consistency = make_judge(
    name="factual_consistency",
    model=JUDGE_MODEL,
    instructions=(
        "Evaluate whether the response is factually consistent with the provided context. "
        "The response should not introduce facts that contradict or are absent from the context.\n\n"
        "Inputs (contains `question` and `context`): {{ inputs }}\n\n"
        "Response: {{ outputs }}"
    ),
)

professionalism = make_judge(
    name="professionalism",
    model=JUDGE_MODEL,
    instructions=(
        "Evaluate the professionalism of the response. A professional response uses formal "
        "language, avoids slang, is well-structured, and maintains a neutral tone.\n\n"
        "Response: {{ outputs }}"
    ),
)

correctness = make_judge(
    name="correctness",
    model=JUDGE_MODEL,
    instructions=(
        "Evaluate whether the response is factually correct based on the expected answer.\n\n"
        "Response: {{ outputs }}\n\n"
        "Expected answer: {{ expectations }}"
    ),
)

custom_judges = [factual_consistency, professionalism, correctness]
print(f"Built {len(custom_judges)} custom LLM-as-Judge scorers")

## 6. Define DeepEval Scorers (via Bedrock)

DeepEval provides advanced RAG + safety metrics. MLflow's `DeepEvalScorer` wrapper integrates them into the same evaluation run, so all results land in a single MLflow run alongside the other scorers.

Key advantage over MLflow's built-in scorers: DeepEval decomposes responses into individual claims and verifies each one — more rigorous than single yes/no judgments.

In [8]:
# MLflow's DeepEval integration lives in mlflow.genai.scorers.deepeval
# (added in MLflow 3.8.0). Each metric is its own class — we instantiate one
# scorer per metric, all using the same Bedrock judge model.
from mlflow.genai.scorers.deepeval import (
    AnswerRelevancy,
    Bias,
    ContextualRelevancy,
    Faithfulness,
    Hallucination,
    Toxicity,
)

deepeval_scorers = [
    Faithfulness(model=JUDGE_MODEL),
    AnswerRelevancy(model=JUDGE_MODEL),
    ContextualRelevancy(model=JUDGE_MODEL),
    Hallucination(model=JUDGE_MODEL),
    Toxicity(model=JUDGE_MODEL),
    Bias(model=JUDGE_MODEL),
]

print(f"Built {len(deepeval_scorers)} DeepEval scorers using {JUDGE_MODEL}")

Built 6 DeepEval scorers using bedrock:/us.anthropic.claude-sonnet-4-5-20250929-v1:0


## 7. Assemble the Full Scorer List

Combine MLflow built-ins, Guidelines, custom judges, DeepEval, and code-based scorers.

In [9]:
from mlflow.genai.scorers import (
    Equivalence,
    Fluency,
    Guidelines,
    RelevanceToQuery,
)

scorers_list = [
    # MLflow built-in LLM-as-Judge
    RelevanceToQuery(model=JUDGE_MODEL),
    Equivalence(model=JUDGE_MODEL),
    Fluency(model=JUDGE_MODEL),
    # MLflow Guidelines-based
    Guidelines(
        name="answer_groundedness",
        guidelines="The answer must be grounded in the provided context and not hallucinate facts.",
        model=JUDGE_MODEL,
    ),
    Guidelines(
        name="answer_completeness",
        guidelines="The answer must fully address the question without omitting key information.",
        model=JUDGE_MODEL,
    ),
    # Custom LLM-as-Judge
    *custom_judges,
    # DeepEval
    *deepeval_scorers,
    # Code-based
    exact_match,
    is_concise,
    word_overlap,
    response_length,
]

print(f"Total scorers registered: {len(scorers_list)}")

Total scorers registered: 18


## 8. Build the Bedrock Predict Function

`mlflow.genai.evaluate()` calls `predict_fn` once per sample in the dataset. The parameter names (`question`, `context`) must match the keys under `inputs` in `eval_data`.

We build one `predict_fn` per model and pass the correct `modelId` via closure.

In [10]:
def make_predict_fn(model_id: str):
    """Return a predict_fn closed over a specific Bedrock model id.

    Reuses the `bedrock` runtime client created in the setup cell above.
    """

    def predict_fn(question: str, context: str = "") -> str:
        prompt = f"Context: {context}\n\nQuestion: {question}\n\nAnswer concisely."
        try:
            response = bedrock.converse(
                modelId=model_id,
                messages=[{"role": "user", "content": [{"text": prompt}]}],
                inferenceConfig={"maxTokens": 512, "temperature": 0.1},
            )
            return response["output"]["message"]["content"][0]["text"]
        except Exception as e:
            return f"ERROR: {e}"

    return predict_fn


# Quick sanity check — generate one response with the first model
first_key = next(iter(BEDROCK_MODELS))
_sample = make_predict_fn(BEDROCK_MODELS[first_key])(
    question=eval_data[0]["inputs"]["question"],
    context=eval_data[0]["inputs"]["context"][:500],
)
print(f"[{first_key} sample response]\n{_sample[:500]}")

[claude-sonnet-4-5 sample response]
Based on the context provided, it describes **voluntary immigration** of African nationals to the United States, particularly after the Immigration and Nationality Act of 1965.

However, this is distinct from how most African Americans' ancestors arrived: the majority descended from Africans who were **forcibly brought as enslaved people** during the transatlantic slave trade (roughly 1619-1865), not through voluntary immigration.

The context discusses modern African immigrants, not the histori


## 9. Run the Evaluation

We now run `mlflow.genai.evaluate()` once per model, each inside its own MLflow run. This gives us side-by-side runs in the MLflow UI for direct comparison.

In [ ]:
import mlflow
import mlflow.genai

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

run_ids = {}

for model_key, model_id in BEDROCK_MODELS.items():
    print(f"\n=== Evaluating {model_key} ({model_id}) ===")
    with mlflow.start_run(run_name=f"eval-{model_key}") as run:
        mlflow.log_param("model_id", model_id)
        mlflow.log_param("model_key", model_key)
        mlflow.log_param("dataset", DATASET_NAME)
        mlflow.log_param("sample_size", len(eval_data))

        results = mlflow.genai.evaluate(
            data=eval_data,
            predict_fn=make_predict_fn(model_id),
            scorers=scorers_list,
        )

        run_ids[model_key] = run.info.run_id
        print(f"Run {run.info.run_id} finished. Metrics: {list(results.metrics.keys())[:5]}...")

## 10. Log Per-Sample Artifacts

The `results` object above contains aggregated metrics. For deeper inspection, we also log a CSV of per-sample ground truth and a JSON of aggregated metrics as artifacts on each run.

In [12]:
summary_df = pd.DataFrame(
    [
        {
            "question": d["inputs"]["question"],
            "ground_truth": d["expectations"]["expected_response"],
            "context": d["inputs"].get("context", "")[:200],
        }
        for d in eval_data
    ]
)
summary_df.to_csv("/tmp/eval_inputs.csv", index=False)

for model_key, run_id in run_ids.items():
    with mlflow.start_run(run_id=run_id):
        mlflow.log_artifact("/tmp/eval_inputs.csv", artifact_path="eval_tables")

print("Artifacts logged for all runs")

Artifacts logged for all runs


## 11. Compare Models Side-by-Side

Query MLflow for all runs in our experiment and display key metrics in a single dataframe.

In [13]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
runs = mlflow.search_runs(experiment_names=[EXPERIMENT_NAME], order_by=["start_time DESC"])

metric_cols = [c for c in runs.columns if c.startswith("metrics.")]
param_cols = [c for c in runs.columns if c in ("params.model_key", "params.sample_size")]

display_cols = ["run_id", "status"] + param_cols + metric_cols
runs[display_cols].head(10)

,run_id,status,params.model_key,params.sample_size,metrics.Faithfulness/mean,metrics.ContextualRelevancy/mean,metrics.Toxicity/mean,metrics.AnswerRelevancy/mean,metrics.fluency/mean,metrics.response_length/mean,metrics.answer_completeness/mean,metrics.Hallucination/mean,metrics.answer_groundedness/mean,metrics.Bias/mean,metrics.relevance_to_query/mean,metrics.word_overlap/mean,metrics.exact_match/mean,metrics.equivalence/mean,metrics.is_concise/mean
0,23575c548eaf4d7c8c0114d324e909a8,FINISHED,claude-haiku-4-5,5,1.0,0.0,1.0,1.0,1.0,83.0,0.8,1.0,1.0,1.0,1.0,0.496392,0.0,0.4,0.8
1,ce9223dba2bf41c48a9933560999a55d,FINISHED,claude-sonnet-4-5,5,1.0,0.0,1.0,1.0,1.0,69.4,0.6,1.0,1.0,1.0,1.0,0.375883,0.0,0.2,0.8


## 12. Open the MLflow UI

### Local MLflow

```bash
mlflow ui --backend-store-uri ./mlruns
```

Then open http://localhost:5000.

### SageMaker-managed MLflow App

Use the `get_mlflow_app_ui_url()` helper defined above — it wraps the `sagemaker:CreatePresignedMlflowAppUrl` API and reuses the `MLFLOW_TRACKING_URI` (App ARN) already resolved in this notebook. Run the next cell to open a fresh presigned UI URL.

In the UI you can:
- Compare the two model runs side-by-side
- Drill into individual scorer results per sample
- Download artifacts (the CSV logged above)
- Track metric trends across future experiments

In [14]:
from IPython.display import HTML, display

# Generate a fresh presigned URL for the MLflow App UI using the boto3 SDK.
# Equivalent CLI (for reference):
#   aws sagemaker create-presigned-mlflow-app-url --arn <MLFLOW_TRACKING_URI> --region <region>
#
# Only meaningful when MLFLOW_TRACKING_URI is an MLflow App ARN — the local
# ./mlruns store is opened via `mlflow ui` instead.
if MLFLOW_TRACKING_URI.startswith("arn:aws:sagemaker:"):
    ui_url = get_mlflow_app_ui_url(MLFLOW_TRACKING_URI, expires_in=300)
    display(
        HTML(
            f'<a href="{ui_url}" target="_blank" rel="noopener noreferrer">'
            f'Open MLflow UI (valid 5 min)</a>'
        )
    )
else:
    print(
        f"MLFLOW_TRACKING_URI is local ({MLFLOW_TRACKING_URI}). "
        f"Start the local UI with: mlflow ui --backend-store-uri {MLFLOW_TRACKING_URI}"
    )

## Summary

In this notebook you:

1. Configured MLflow (local or SageMaker-managed) and Bedrock credentials
2. Loaded and reshaped the `ragas-wikiqa` dataset into the `{inputs: {question, context}, expectations}` RAG-generation format
3. Registered **18 scorers** across five categories (code-based, MLflow built-in, Guidelines, `make_judge`, DeepEval), including RAG-grounding scorers (`Faithfulness`, `Hallucination`, `answer_groundedness`, `factual_consistency`)
4. Ran `mlflow.genai.evaluate()` against two Bedrock models using Bedrock Claude as the judge
5. Logged parameters, metrics, and artifacts for each run
6. Compared the two models side-by-side via `mlflow.search_runs`

Remember: this evaluates the **generator** of a RAG pipeline, holding retrieval constant. To evaluate retrieval quality, you'd need a retriever emitting top-k contexts and metrics like recall@k / precision@k / MRR.

### Next Steps

- Swap in your own dataset (any list of `{inputs, expectations}` dicts will work)
- Add additional `make_judge` scorers for domain-specific criteria (tone, compliance, style)
- Point `MLFLOW_TRACKING_URI` at a SageMaker MLflow App for team-wide tracking
- Integrate this evaluation into CI so every prompt or model change runs through the same scorers
- Extend to a full RAG eval by plugging in a retriever and adding retrieval metrics